# Sızıntı analizi

Sızıntı kontrolünde test görüntülerinin büyük kısmının (3.604 görüntünün 2.820'si) train+val havuzunda **çok benzer** (pHash Hamming mesafesi ≤ 2) bir eşi çıktı. Bu notebook bunun gerçek bir veri sızıntısı mı yoksa MRI'daki siyah arka planın pHash'i yanıltması mı olduğunu ayırt eder. GPU gerekmez.

1. Dosya adlarında hasta kimliği veya kesit numarası olup olmadığına bakar. Varsa hasta bazlı bölme yapılabilir.
2. Her test görüntüsü için train+val'de en yakın görüntüyü bulur ve **en yakın görüntünün etiketiyle test etiketinin ne sıklıkla aynı olduğunu** ölçer. Bu, hiç eğitim yapmadan elde edilen bir doğruluktur.
3. En benzer çiftlerden 8 tanesini yan yana gösterir.

Sonuçları (tabloyu, dosya adı örneklerini ve görsel çiftleri) Claude'a gönder.


In [ ]:
# ======================= SIZINTI ANALİZİ (GPU gerekmez, ~10 dk) =======================
# Soru: test görüntülerinin train+val'de "çok benzer" bir eşi varsa, bu eşler aynı etiketi mi taşıyor?
# Eğer pHash'e göre en yakın train görüntüsünün etiketi test etiketini çok yüksek oranda tahmin ediyorsa,
# modeller lezyonu değil "aynı hastanın neredeyse aynı kesitini" öğreniyor olabilir (veri sızıntısı).
import subprocess, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
try:
    import imagehash
except ImportError:
    subprocess.run(["pip", "install", "-q", "imagehash"], check=True); import imagehash
from google.colab import drive
drive.mount("/content/drive")
import kagglehub

KAGGLE_ID = "abenjelloun/breast-mri-tumor-classification-dataset"
OUT = Path("/content/drive/MyDrive/breast_mri_standard_results/leakage_analysis"); OUT.mkdir(parents=True, exist_ok=True)
RAW = Path(kagglehub.dataset_download(KAGGLE_ID))

def collect(split_names):
    rows = []
    for d in RAW.rglob("*"):
        if d.is_dir() and d.name.lower() in split_names:
            for cls in d.iterdir():
                if cls.is_dir() and cls.name.lower() in ("benign", "malignant"):
                    for p in sorted(cls.iterdir()):
                        if p.is_file():
                            rows.append({"split": d.name.lower(), "file": str(p), "name": p.name,
                                         "label": int(cls.name.lower() == "malignant")})
    return pd.DataFrame(rows)

pool = collect({"train", "val", "valid", "validation"}); test = collect({"test"})
print("train+val:", len(pool), "| test:", len(test))

# 1) Dosya adları: hasta kimliği / kesit numarası içeriyor mu?
print("\nÖrnek dosya adları (hasta ID'si var mı diye bak):")
for s, g in pd.concat([pool, test]).groupby("split"):
    print(f"  {s}:", list(g.sample(8, random_state=0).name))

# 2) pHash ve her test görüntüsü için en yakın train+val komşusu
def phash_all(files):
    return np.array([int(str(imagehash.phash(Image.open(f).convert("L"))), 16) for f in files], dtype=np.uint64)
print("\npHash hesaplanıyor...")
P, T = phash_all(pool.file), phash_all(test.file)
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
Pb = P.view(np.uint8).reshape(-1, 8)
nn_idx, nn_dist = np.empty(len(T), int), np.empty(len(T), int)
for i, h in enumerate(T):
    d = POP[Pb ^ np.array([h], dtype=np.uint64).view(np.uint8)].sum(1)
    j = int(d.argmin()); nn_idx[i], nn_dist[i] = j, int(d[j])
test["nn_dist"] = nn_dist
test["nn_label"] = pool.label.values[nn_idx]
test["nn_file"] = pool.file.values[nn_idx]
test["same_label"] = test.label == test.nn_label

# 3) Özet: en yakın komşunun etiketi test etiketini ne kadar tahmin ediyor? (eğitimsiz "1-NN pHash sınıflandırıcı")
rows = []
for lo, hi, name in [(0, 0, "0 (aynı hash)"), (1, 2, "1–2"), (3, 6, "3–6"), (7, 12, "7–12"), (13, 64, "13+")]:
    g = test[(test.nn_dist >= lo) & (test.nn_dist <= hi)]
    rows.append({"Hamming mesafesi": name, "Test görüntüsü": len(g), "Oran (%)": round(100 * len(g) / len(test), 1),
                 "Komşuyla aynı etiket (%)": round(100 * g.same_label.mean(), 1) if len(g) else None})
summary = pd.DataFrame(rows)
summary.loc[len(summary)] = {"Hamming mesafesi": "TÜMÜ", "Test görüntüsü": len(test), "Oran (%)": 100.0,
                             "Komşuyla aynı etiket (%)": round(100 * test.same_label.mean(), 1)}
display(summary)
summary.to_csv(OUT / "nn_label_agreement.csv", index=False)
test.drop(columns=["file"]).to_csv(OUT / "test_nearest_neighbors.csv", index=False)

print("YORUM: 'TÜMÜ' satırındaki oran, hiç eğitim yapmadan sadece en yakın görüntünün etiketine bakarak elde edilen doğruluktur.")
print("  ~%50 → yakın eşler etiket bilgisi taşımıyor (pHash benzerliği büyük ihtimalle siyah arka plandan; sızıntı zayıf).")
print("  %90+ → test görüntülerinin büyük kısmı train'deki neredeyse aynı görüntüyle eşleşiyor (güçlü sızıntı).")

# 4) Gözle kontrol: en yakın 8 çift (test | en yakın train+val)
show = test[test.nn_dist <= 2].sample(min(8, (test.nn_dist <= 2).sum()), random_state=1)
if len(show):
    fig, ax = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)))
    ax = np.atleast_2d(ax)
    lab = {0: "Benign", 1: "Malignant"}
    for r, (_, row) in enumerate(show.iterrows()):
        ax[r, 0].imshow(Image.open(row.file).convert("L"), cmap="gray"); ax[r, 0].set_title(f"TEST: {lab[row.label]}\n{row['name']}", fontsize=8)
        ax[r, 1].imshow(Image.open(row.nn_file).convert("L"), cmap="gray"); ax[r, 1].set_title(f"EN YAKIN: {lab[row.nn_label]} (d={row.nn_dist})\n{Path(row.nn_file).name}", fontsize=8)
        for a in ax[r]: a.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "nearest_pairs.png", dpi=150); plt.show()
